In [55]:
import os
import json
import re
import pandas as pd
import torch
import numpy as np

from utils import exact_match_results, relax_match_results


In [56]:
# model_folder = 'Output_gemma4_26b_no_sim'
# model_folder = "Output_llama3.1_8b_no_sim"
model_folder = "Output_mistral-small_24b_no_sim"
# model_folder = "Output_gemma4_31b_no_sim"

# model_sim_folder = 'Output_llama3.1_8b_sim'
# model_sim_folder = "Output_gemma4_31b_sim"
model_sim_folder = "Output_mistral-small_24b_sim"
# model_sim_folder = "Output_gemma4_26b_sim"

if os.path.exists(f"{model_sim_folder}") == False:
    os.makedirs(f"{model_sim_folder}")

if os.path.exists(f"{model_sim_folder}") == False:
    os.makedirs(f"{model_sim_folder}")

In [57]:
# Checking if there's any null output:
for sample in os.listdir(model_folder):
    with open(f"{model_folder}/{sample}", "r") as f:
        data = json.load(f)
        if not data:
            print(f"Sample {sample} has null output.")


In [58]:
preds_dict_od_all = []
preds_dict_all_temp = []
for sample in os.listdir(model_folder):
    # if sample in ['sample_68', 'sample_635', 'sample_71', 'sample_324','sample_2748']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()
        preds_dict_all_temp.append(preds_dict)
        preds_dict_processed = []
        preds_dict_od = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:

                    preds_dict_od.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                    preds_dict_od_all.append({'sample':sample,'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
        # with open(f"{model_attribution_folder}/{sample}", 'w') as f:
        #     json.dump(preds_dict_od, f, indent=2)

len(preds_dict_od_all)

evaluating sample_38_20210610044413_Hayden:
evaluating sample_1765:
evaluating sample_160_20210112211715:
evaluating sample_1231:
evaluating sample_60_20210604143820_Isra:
evaluating sample_50_20210604143818_Isra:
evaluating sample_154_20210112211713:
evaluating sample_76_20210112211704:
evaluating sample_364:
evaluating sample_963:
evaluating sample_166_20210112211715:
evaluating sample_85_20210112211705:
evaluating sample_5_20210604143806_Hayden:
evaluating sample_2765:
evaluating sample_2762:
evaluating sample_36_20210610044412_Hayden:
evaluating sample_30_20210610044412_Isra:
evaluating sample_367:
evaluating sample_1433:
evaluating sample_1600:
evaluating sample_1854:
evaluating sample_2_20210604143805_Hayden:
evaluating sample_34_20210610044412_Hayden:
evaluating sample_225:
evaluating sample_58_20210604143820_Isra:
evaluating sample_2748:
evaluating sample_2780:
evaluating sample_90_20210604143828_Booma:
evaluating sample_1240:
evaluating sample_96_20210112211707:
evaluating sam

25

# Similarity algorithm

In [59]:
# from sentence_transformers import SentenceTransformer, util
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# for sample in os.listdir(model_folder):
#     # if sample == 'sample_58_20210604143820_Isra':
#     #Read original text
#     #Read label dict
#         print(f"processing: {sample}")

#         with open(f'label/{sample}.json', 'r') as f:
#             label_dict = json.load(f)
            
#         #Read output
#         with open(f"{model_folder}/{sample}", 'r') as f:
#             org_preds_json = json.load(f)

#         #Read original text
#         with open(f'notes/{sample}.txt', 'r') as file:
#             text = file.read()

#         preds_dict = []
#         messup_pred_tokens = []
#         for ent in org_preds_json:
#             for phrase, label in ent.items():
#                 matches = list(re.finditer(re.escape(phrase), text, re.IGNORECASE))
#                 if matches:
#                     for match in matches:
#                         # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
#                         preds_dict.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
#                 else:
#                     messup_pred_tokens.append(phrase)
        
        
#         clean_text = re.sub(r'[.,:&]', '', text)
#         clean_original_tokens = clean_text.split()

#         # Encode all candidate spans
#         n = len(clean_original_tokens)
#         print(f"length of the text: {n}")


#         spans_dup = []
#         for i in range(0, n):
#             for w in range(1, n+1):
#                 spans_dup.append(' '.join(clean_original_tokens[i:i+w]))
#         spans = list(set(spans_dup)) #Deduplication

#         #Embedding
#         spans_emb = model.encode(spans)

#         alg_pred_tokens = []
#         for token in messup_pred_tokens:    
#             # Encode target
#             target_emb = model.encode(token)
#             sim = util.cos_sim(target_emb, spans_emb) 

#             #Mapping back all values = highest value
#             top_idx = sim.argmax()

#             #Map back embedded vector to tokens
#             best_match = spans[top_idx]
#             alg_pred_tokens.append(best_match)

#         #Write back the output
#         mapping = dict(zip(messup_pred_tokens, alg_pred_tokens))

#         original = []
#         updated = []

#         for record in org_preds_json:
#             original_record = {}
#             updated_record = {}

#             for key, value in record.items():

#                 if key in mapping:
#                     original_record[key] = "false positive"
#                     updated_record[mapping[key]] = value
                    
#                 else:
#                     original_record[key] = value
#                     updated_record[key] = value

#             original.append(original_record)
#             updated.append(updated_record)
        
#         # # with open(f"{model_no_sim_folder}/{sample}", 'w') as f:
#         # #     json.dump(original, f, indent=2)
#         # with open(f"{model_sim_folder}/{sample}", 'w') as f:
#         #     json.dump(updated, f, indent=2)


In [60]:
from sentence_transformers import SentenceTransformer, util
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
print(f"Using device: {device}")

for sample in os.listdir(model_folder):
    # if sample == 'sample_134_20210112211711':
    #Read original text
    #Read label dict
        print(f"processing: {sample}")

        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)
            
        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            org_preds_json = json.load(f)

        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict = []
        messup_pred_tokens = []
        for ent in org_preds_json:
            for phrase, label in ent.items():
                matches = list(re.finditer(re.escape(phrase), text, re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    messup_pred_tokens.append(phrase)
        
        # print(messup_pred_tokens)
        
        clean_text = re.sub(r'[.,:&]', '', text)
        clean_original_tokens = clean_text.split()

        # Encode all candidate spans
        n = len(clean_original_tokens)
        print(f"length of the text: {n}")


        unique_messup_pred_tokens = list(dict.fromkeys(messup_pred_tokens))
        max_target_length = max((len(token.split()) for token in unique_messup_pred_tokens if token.strip()), default=0)
        target_lengths = range(1, max_target_length*5)

        spans = []
        seen_spans = set()
        for i in range(n):
            remaining = n - i
            for width in target_lengths:
                if width > remaining:
                    break
                span = ' '.join(clean_original_tokens[i:i + width])
                if span not in seen_spans:
                    seen_spans.add(span)
                    spans.append(span)

        if spans and unique_messup_pred_tokens:
            spans_emb = model.encode(
                spans,
                batch_size=128,
                convert_to_tensor=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )

            target_emb = model.encode(
                unique_messup_pred_tokens,
                batch_size=128,
                convert_to_tensor=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )

            sim = util.cos_sim(target_emb, spans_emb)
            top_indices = sim.argmax(dim=1).tolist()
            resolved_tokens = [spans[idx] for idx in top_indices]
            resolved_mapping = dict(zip(unique_messup_pred_tokens, resolved_tokens))
            alg_pred_tokens = [resolved_mapping[token] for token in messup_pred_tokens]
        else:
            alg_pred_tokens = []

        #Write back the output
        mapping = dict(zip(messup_pred_tokens, alg_pred_tokens))

        original = []
        updated = []

        for record in org_preds_json:
            original_record = {}
            updated_record = {}

            for key, value in record.items():

                if key in mapping:
                    original_record[key] = "false positive"
                    updated_record[mapping[key]] = value
                    
                else:
                    original_record[key] = value
                    updated_record[key] = value

            original.append(original_record)
            updated.append(updated_record)
        
        # # with open(f"{model_no_sim_folder}/{sample}", 'w') as f:
        # #     json.dump(original, f, indent=2)
        with open(f"{model_sim_folder}/{sample}", 'w') as f:
            json.dump(updated, f, indent=2)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18618.05it/s]


Using device: cuda
processing: sample_38_20210610044413_Hayden
length of the text: 540
processing: sample_1765
length of the text: 368
processing: sample_160_20210112211715
length of the text: 575
processing: sample_1231
length of the text: 223
processing: sample_60_20210604143820_Isra
length of the text: 466
processing: sample_50_20210604143818_Isra
length of the text: 602
processing: sample_154_20210112211713
length of the text: 383
processing: sample_76_20210112211704
length of the text: 241
processing: sample_364
length of the text: 556
processing: sample_963
length of the text: 366
processing: sample_166_20210112211715
length of the text: 749
processing: sample_85_20210112211705
length of the text: 457
processing: sample_5_20210604143806_Hayden
length of the text: 406
processing: sample_2765
length of the text: 243
processing: sample_2762
length of the text: 1003
processing: sample_36_20210610044412_Hayden
length of the text: 413
processing: sample_30_20210610044412_Isra
length of

In [61]:
tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_sim_folder):
    # if sample not in ['sample_140_20210112211712']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_sim_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + rm_mm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + em_mm + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall)}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall)}")

evaluating sample_38_20210610044413_Hayden:
[{'entity': 'pancreatic mass', 'label': 'PROBLEM', 'start': 330, 'end': 345}, {'entity': 'pancreatic mass', 'label': 'PROBLEM', 'start': 781, 'end': 796}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 360, 'end': 371}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 811, 'end': 822}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 3226, 'end': 3237}, {'entity': 'strictured pancreatic duct', 'label': 'PROBLEM', 'start': 1054, 'end': 1080}, {'entity': 'nausea', 'label': 'PROBLEM', 'start': 1546, 'end': 1552}, {'entity': 'CT scan', 'label': 'TEST', 'start': 290, 'end': 297}, {'entity': 'CT scan', 'label': 'TEST', 'start': 732, 'end': 739}, {'entity': 'ERCP', 'label': 'TEST', 'start': 3009, 'end': 3013}, {'entity': 'stent', 'label': 'TREATMENT', 'start': 404, 'end': 409}, {'entity': 'stent', 'label': 'TREATMENT', 'start': 855, 'end': 860}, {'entity': 'stent', 'label': 'TREATMENT', 'start': 1031, 'end': 1036}, {'entity': 'st

In [62]:
tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_folder):
    # if sample not in ['sample_140_20210112211712']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        # print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + rm_mm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + em_mm + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall)}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall)}")

evaluating sample_38_20210610044413_Hayden:
[{'entity': 'pancreatic mass', 'label': 'PROBLEM', 'start': 330, 'end': 345}, {'entity': 'pancreatic mass', 'label': 'PROBLEM', 'start': 781, 'end': 796}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 360, 'end': 371}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 811, 'end': 822}, {'entity': 'lymph nodes', 'label': 'PROBLEM', 'start': 3226, 'end': 3237}, {'entity': 'strictured pancreatic duct', 'label': 'PROBLEM', 'start': 1054, 'end': 1080}, {'entity': 'nausea', 'label': 'PROBLEM', 'start': 1546, 'end': 1552}, {'entity': 'CT scan', 'label': 'TEST', 'start': 290, 'end': 297}, {'entity': 'CT scan', 'label': 'TEST', 'start': 732, 'end': 739}, {'entity': 'ERCP', 'label': 'TEST', 'start': 3009, 'end': 3013}, {'entity': 'stent', 'label': 'TREATMENT', 'start': 404, 'end': 409}, {'entity': 'stent', 'label': 'TREATMENT', 'start': 855, 'end': 860}, {'entity': 'stent', 'label': 'TREATMENT', 'start': 1031, 'end': 1036}, {'entity': 'st